# C12 — Famille A sur les émotions : CNN et CRNN sur le log-Mel (Brique 4, P1, P6, P8, E4)

Les réseaux de C9 (6 sorties au lieu de 50), sur CREMA-D et RAVDESS, avec le découpage par
acteur de C4 (P1) et les 6 classes communes (P2) :

1. log-Mel de chaque extrait entier calculé **une seule fois** (R5, R6), gardé en sortie ;
2. fenêtre fixe de 3 s (301 trames) : tirée au hasard dans l'extrait au train, centrale en
   validation et en test, complétée par du silence si l'extrait est plus court ;
3. standardisation par bande sur le train seul (R4), augmentations du train (P6) ;
4. taux d'apprentissage choisi sur la validation (graine 42), puis graines 42, 43, 44 : chaque
   graine suit la courbe complète de 100 époques et garde sa meilleure époque de validation ;
5. test utilisé une seule fois : scores globaux, par jeu, par genre, par acteur (E4) ; McNemar
   avec correction de Holm (P8) ; comparaison avec la tête whisper-base de C11 (H1, H4) ;
6. le meilleur réseau (choisi sur la validation) devient le modèle d'émotion du profil temps
   réel, calibré (E2).

## a. Démarrage

`SMOKE_TEST = True` fait un essai à blanc (2 époques) pour vérifier le notebook, jamais pour des
résultats.

In [ ]:
import sys

BRANCH = "main"
SMOKE_TEST = False
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## b. ffmpeg et GPU

In [ ]:
import shutil
import time

import torch

if shutil.which("ffmpeg") is None:
    !apt-get -qq update && apt-get -qq install -y ffmpeg
if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU : activer l'accélérateur GPU T4 du notebook.")
device = torch.device("cuda", 0)
GPU_NAME = torch.cuda.get_device_name(device)
print("GPU :", GPU_NAME, "· torch", torch.__version__)
NOTEBOOK_START = time.time()

## c. Trouver CREMA-D et RAVDESS

Comme C10 : CREMA-D est le dossier `AudioWAV` qui contient exactement 7 442 WAV ; RAVDESS, le
dossier qui contient `Actor_01` à `Actor_24`, chaque fichier une seule fois. Liens créés dans
`/kaggle/working/datasets/<id du jeu>` (contrat des manifestes).

In [ ]:
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")
DATASETS_DIR = Path("/kaggle/working/datasets")
CREMAD_EXPECTED_WAV = 7442
RAVDESS_EXPECTED_WAV = 1440


def find_cremad_root() -> Path:
    for audio_folder in sorted(INPUT_DIR.glob("**/AudioWAV")):
        wav_count = len(list(audio_folder.glob("*.wav")))
        if wav_count == CREMAD_EXPECTED_WAV:
            return audio_folder.parent
    raise RuntimeError("CREMA-D introuvable : attacher le dataset ejlok1/cremad.")


def find_ravdess_root() -> Path:
    candidates = sorted({actor.parent for actor in INPUT_DIR.glob("**/Actor_01")})
    for candidate in candidates:
        wav_paths = list(candidate.glob("Actor_*/*.wav"))
        unique_names = {path.name for path in wav_paths}
        if len(wav_paths) == RAVDESS_EXPECTED_WAV and len(unique_names) == RAVDESS_EXPECTED_WAV:
            return candidate
    raise RuntimeError("RAVDESS introuvable : attacher uwrfkaggler/ravdess-emotional-speech-audio.")


DATASETS_DIR.mkdir(parents=True, exist_ok=True)
for dataset_id, root in [("cremad", find_cremad_root()), ("ravdess", find_ravdess_root())]:
    link = DATASETS_DIR / dataset_id
    if link.is_symlink():
        link.unlink()
    link.symlink_to(root)
    print(dataset_id, "→", root)

## d. Manifestes, découpage par acteur (C4) et contrôles

Le découpage versionné `data/splits/emotion_split.csv` répartit les **acteurs** (P1). On
vérifie qu'aucun acteur n'est dans deux parties, puis, après le calcul, qu'aucun extrait n'a de
classe hors des 6 classes communes (P2).

In [ ]:
from echolens.config import CONFIG_DIR, REPO_ROOT
from echolens.core.data.emotions import load_emotion_labels
from echolens.core.data.manifest import load_manifest
from echolens.core.data.splits import check_no_speaker_leak, load_speaker_splits, load_split_config

split_config = load_split_config()
speaker_splits = load_speaker_splits(REPO_ROOT / "data" / "splits" / "emotion_split.csv")
check_no_speaker_leak(speaker_splits)
emotion_labels = load_emotion_labels()
rows_by_dataset = {}
for dataset_id in split_config.training_emotion_datasets:
    rows_by_dataset[dataset_id] = load_manifest(
        REPO_ROOT / "data" / "manifests" / f"{dataset_id}.csv"
    )
    print(dataset_id, ":", len(rows_by_dataset[dataset_id]), "lignes de manifeste")
print("Aucun acteur dans deux parties du découpage.")

## e. Log-Mel des extraits, calculés une seule fois (R5, R6)

Chaque extrait gardé passe par `load_audio` puis `echolens.core.features.log_mel_spectrogram`.
Le log-Mel entier est gardé (avec sa longueur), complété jusqu'au plus long par du silence ; la
fenêtre de 3 s est découpée plus tard. Puis : effectifs par partie, et part moyenne de silence
ajouté par jeu (extraits de moins de 3 s).

In [ ]:
from collections import Counter

import numpy as np

from echolens.core.features import FeatureConfig
from echolens.core.frame_windows import window_frame_count
from echolens.training.family_a import load_family_a_config
from echolens.training.family_a_emotions import (
    compute_emotion_log_mels,
    load_emotion_log_mels,
    save_emotion_log_mels,
    silence_shares,
)

OUTPUT_DIR = Path("/kaggle/working")
LOG_MELS_PATH = OUTPUT_DIR / "emotion_log_mels.npz"
feature_config = FeatureConfig.from_yaml(CONFIG_DIR / "features.yaml")
family_a_config = load_family_a_config()
window_frames = window_frame_count(family_a_config.emotions.window_s, feature_config)

start = time.time()
if LOG_MELS_PATH.exists():
    log_mel_set = load_emotion_log_mels(LOG_MELS_PATH)
else:
    log_mel_set = compute_emotion_log_mels(
        rows_by_dataset, DATASETS_DIR, emotion_labels, speaker_splits, split_config, feature_config
    )
    save_emotion_log_mels(log_mel_set, LOG_MELS_PATH)
elapsed = time.time() - start
print(f"Log-Mel : {log_mel_set.log_mels.shape} en {elapsed:.0f} s")
print(f"Fenêtre de {window_frames} trames")

# Contrôle P1 : chaque acteur dans une seule partie, sur les extraits réellement gardés.
actors_by_split = {}
for index, split in enumerate(log_mel_set.splits):
    actor = f"{log_mel_set.datasets[index]}:{log_mel_set.speakers[index]}"
    actors_by_split.setdefault(split, set()).add(actor)
for first in actors_by_split:
    for second in actors_by_split:
        if first < second and actors_by_split[first] & actors_by_split[second]:
            raise RuntimeError(f"Fuite d'acteurs entre {first} et {second}")
# Contrôle P2 : uniquement les 6 classes communes (calme et surprise exclus).
class_count = len(emotion_labels.emotions)
if log_mel_set.targets.min() < 0 or log_mel_set.targets.max() >= class_count:
    raise RuntimeError("Une classe sort des 6 émotions communes")
counts = Counter(zip(log_mel_set.datasets, log_mel_set.splits, strict=True))
for key in sorted(counts):
    print(key, counts[key], "extraits")
print({split: len(actors) for split, actors in actors_by_split.items()}, "acteurs par partie")
shares = silence_shares(log_mel_set, window_frames)
print(
    "Part moyenne de silence ajouté :",
    {dataset: round(share, 4) for dataset, share in shares.items()},
)

## f. Entraînement du CNN puis du CRNN

Pour chaque réseau : un entraînement par taux d'apprentissage (graine 42) sur 100 époques,
F1 macro de validation après chaque époque ; le meilleur taux est retenu. Puis les graines 42,
43, 44 avec ce taux, chacune sur la courbe complète, en gardant sa meilleure époque. Les
probabilités de validation et de test de chaque graine sont écrites dans `c12/arrays/`.

In [ ]:
from echolens.training.family_a_emotions_training import emotion_parts, run_emotion_architecture

ARCHITECTURES = ["cnn", "crnn"]
ARRAYS_DIR = OUTPUT_DIR / "c12" / "arrays"
ARRAYS_DIR.mkdir(parents=True, exist_ok=True)
if SMOKE_TEST:
    smoke_emotions = family_a_config.emotions.model_copy(update={"max_epochs": 2})
    family_a_config = family_a_config.model_copy(update={"emotions": smoke_emotions})

parts = emotion_parts(log_mel_set, window_frames, feature_config.db_floor)
runs = {}
gpu_seconds = {}
for architecture in ARCHITECTURES:
    start = time.time()
    run = run_emotion_architecture(architecture, parts, class_count, family_a_config, device)
    gpu_seconds[architecture] = time.time() - start
    runs[architecture] = run
    arrays = {"validation_indices": parts.validation_indices, "test_indices": parts.test_indices}
    for seed_run in run.runs:
        arrays[f"seed{seed_run.seed}_validation_probabilities"] = seed_run.validation.probabilities
        arrays[f"seed{seed_run.seed}_test_probabilities"] = seed_run.test.probabilities
        if seed_run.test.attention is not None:
            arrays[f"seed{seed_run.seed}_test_attention"] = seed_run.test.attention
    np.savez(ARRAYS_DIR / f"{architecture}.npz", **arrays)
    for trial in run.trials:
        score = trial.best_validation_f1
        learning_rate = trial.learning_rate
        print(
            f"{architecture} lr {learning_rate:g} : époque {trial.best_epoch}, F1 val {score:.3f}"
        )
    print(f"{architecture} : lr retenu {run.chosen_learning_rate:g}")
    for seed_run in run.runs:
        score = seed_run.best_validation_f1
        print(f"  graine {seed_run.seed} : époque {seed_run.best_epoch}, F1 val {score:.3f}")
    print(f"  {gpu_seconds[architecture] / 60:.1f} min")

## g. Résultats de test, McNemar et comparaison avec C11

Le test est prédit une seule fois par graine. Moyenne et écart-type (ddof = 0) sur les 3
graines ; intervalle bootstrap et McNemar sur la graine 42. La tête whisper-base de C11
(prédictions de test de la graine 42, PR #24) est alignée extrait par extrait sur le même test :
on vérifie d'abord qu'on retrouve son F1. Les p-valeurs de McNemar sont corrigées par Holm.

In [ ]:
import datetime
import json
import subprocess

from echolens.training.evaluation import (
    EvaluationConfig,
    bootstrap_ci,
    holm_correction,
    macro_f1,
    normalized_confusion,
)
from echolens.training.family_a_emotions import (
    actor_spread,
    clip_predictions,
    seed_summary,
    seed_test_scores,
)
from echolens.training.family_a_models import predicted_classes
from echolens.training.family_a_report import compare_models

evaluation_config = EvaluationConfig.from_yaml(CONFIG_DIR / "evaluation.yaml")
commit = subprocess.run(
    ["git", "-C", CLONE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
test_indices = parts.test_indices
y_true = log_mel_set.targets[test_indices]
REFERENCE_SEED = 42

reports = {}
reference_predictions = {}
for architecture, run in runs.items():
    per_seed = []
    for seed_run in run.runs:
        y_pred = predicted_classes(seed_run.test)
        per_seed.append(seed_test_scores(log_mel_set, test_indices, y_pred, class_count))
        if seed_run.seed == REFERENCE_SEED:
            reference_predictions[architecture] = clip_predictions(
                log_mel_set, test_indices, y_pred
            )
            low, high = bootstrap_ci(y_true, y_pred, evaluation_config)
            confusion = normalized_confusion(y_true, y_pred, class_count)
    summary = seed_summary(per_seed)
    reports[architecture] = {
        "experiment": f"emotions_family_a_{architecture}",
        "smoke_test": SMOKE_TEST,
        "date": datetime.datetime.now(datetime.UTC).isoformat(),
        "branch": BRANCH,
        "commit": commit,
        "gpu": GPU_NAME,
        "gpu_seconds": gpu_seconds[architecture],
        "classes": emotion_labels.emotions,
        "window_frames": window_frames,
        "silence_share_by_dataset": shares,
        "test_size": len(test_indices),
        "chosen_learning_rate": run.chosen_learning_rate,
        "tuning": [
            {
                "learning_rate": trial.learning_rate,
                "best_epoch": trial.best_epoch,
                "best_validation_macro_f1": trial.best_validation_f1,
                "validation_macro_f1_per_epoch": trial.validation_f1_history,
            }
            for trial in run.trials
        ],
        "seeds": [
            {
                "seed": seed_run.seed,
                "best_epoch": seed_run.best_epoch,
                "best_validation_macro_f1": seed_run.best_validation_f1,
                "test": scores,
            }
            for seed_run, scores in zip(run.runs, per_seed, strict=True)
        ],
        "summary": summary,
        "actor_spread": actor_spread(log_mel_set, test_indices, summary["macro_f1_by_actor"]),
        "reference_seed": REFERENCE_SEED,
        "macro_f1_ci": [low, high],
        "confusion": confusion.tolist(),
        "family_a_config": family_a_config.model_dump(),
    }
    f1 = summary["macro_f1"]
    print(
        f"{architecture} : F1 macro {f1['mean']:.3f} ± {f1['std']:.3f}, IC [{low:.3f} ; {high:.3f}]"
    )

# Tête whisper-base de C11, graine 42, mêmes extraits de test dans le même ordre (jeu, manifeste).
c11_test = sorted(INPUT_DIR.glob("**/c11/emotions/whisper-base/test_seed42.npz"))
if len(c11_test) != 1:
    raise RuntimeError("Attacher la sortie de echolens-c11-family-b-heads (C11)")
with np.load(c11_test[0]) as content:
    c11_targets = content["targets"]
    c11_predicted = content["probabilities"].argmax(axis=1)
if not np.array_equal(c11_targets, y_true):
    raise RuntimeError("Le test de C11 n'a pas les mêmes extraits ou le même ordre")
whisper_predictions = clip_predictions(log_mel_set, test_indices, c11_predicted)
print(f"Tête whisper-base de C11 (graine 42) : F1 macro {macro_f1(y_true, c11_predicted):.4f}")

In [ ]:
# Le meilleur réseau est choisi sur la VALIDATION (moyenne des 3 graines), jamais sur le test.
validation_f1 = {}
for architecture, run in runs.items():
    validation_f1[architecture] = float(
        np.mean([seed_run.best_validation_f1 for seed_run in run.runs])
    )
best = max(validation_f1, key=validation_f1.get)
print("F1 de validation moyen :", validation_f1, "→ meilleur :", best)

comparisons = [
    compare_models(
        "cnn",
        reference_predictions["cnn"],
        "crnn",
        reference_predictions["crnn"],
        evaluation_config,
    ),
    compare_models(
        best, reference_predictions[best], "whisper-base", whisper_predictions, evaluation_config
    ),
]
corrected = holm_correction([comparison["p_value"] for comparison in comparisons])
for comparison, p_holm in zip(comparisons, corrected, strict=True):
    comparison["p_value_holm"] = p_holm
    names = f"{comparison['model_a']} / {comparison['model_b']}"
    scores = f"F1 {comparison['macro_f1_a']:.3f} / {comparison['macro_f1_b']:.3f}"
    disagreements = f"{comparison['only_a_right']} contre {comparison['only_b_right']}"
    print(
        f"{names} : {scores}, {disagreements}, p = {comparison['p_value']:.3g}, Holm {p_holm:.3g}"
    )
comparison_report = {
    "best_family_a": best,
    "validation_macro_f1": validation_f1,
    "comparisons": comparisons,
}
for architecture, report in reports.items():
    (OUTPUT_DIR / f"emotions_family_a_{architecture}.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"
    )
(OUTPUT_DIR / "emotions_comparisons.json").write_text(
    json.dumps(comparison_report, ensure_ascii=False, indent=2), encoding="utf-8"
)

## h. Modèle d'émotion du profil temps réel

Le meilleur réseau, graine 42, à sa meilleure époque de validation. T est réglée sur ses
probabilités de validation ; l'ECE se mesure sur le test, avant et après. Le modèle est écrit en
safetensors avec son `config.json` (fenêtre de 3 s, standardisation, T, empreinte des features).

In [ ]:
from echolens.core.models.production_config import (
    Calibration,
    ProductionModelConfig,
    TrainingRecord,
    config_fingerprint,
)
from echolens.core.models.storage import save_production_model
from echolens.training.production import RoundPredictions, validation_calibration

best_run = runs[best]
for seed_run in best_run.runs:
    if seed_run.seed == REFERENCE_SEED:
        reference_run = seed_run
validation_indices = parts.validation_indices
validation = RoundPredictions(
    log_mel_set.targets[validation_indices],
    reference_run.validation.probabilities.astype(np.float64),
)
test = RoundPredictions(y_true, reference_run.test.probabilities.astype(np.float64))
outcome = validation_calibration(validation, test, evaluation_config)
model_id = f"echolens-emotion-{best}"
production = ProductionModelConfig(
    model_id=model_id,
    task="emotion",
    architecture=best,
    labels=emotion_labels.emotions,
    standardization_mean=reference_run.trained.statistics.mean.tolist(),
    standardization_std=reference_run.trained.statistics.std.tolist(),
    calibration=Calibration(
        temperature=outcome.temperature,
        method="T réglée sur la validation (graine 42), ECE mesurée sur le test",
        ece_before=outcome.ece_before,
        ece_after=outcome.ece_after,
    ),
    training=TrainingRecord(
        learning_rate=best_run.chosen_learning_rate,
        dropout=family_a_config.network.dropout,
        epochs=reference_run.best_epoch,
        seed=REFERENCE_SEED,
        data="CREMA-D et RAVDESS, découpage par acteur (P1), meilleure époque de validation (C12)",
        commit=commit,
    ),
    window_s=family_a_config.emotions.window_s,
    network=family_a_config.network,
    features_sha256=config_fingerprint(CONFIG_DIR / "features.yaml"),
    license="cc-by-nc-sa-4.0",
)
save_production_model(
    reference_run.trained.network, production, OUTPUT_DIR / "production" / model_id
)
calibration_report = {"model_id": model_id, **production.calibration.model_dump()}
(OUTPUT_DIR / "emotions_production.json").write_text(
    json.dumps(calibration_report, ensure_ascii=False, indent=2), encoding="utf-8"
)
ece_text = f"ECE {outcome.ece_before:.4f} → {outcome.ece_after:.4f}"
print(f"{model_id} : T = {outcome.temperature:.3f}, {ece_text}")

## i. Temps et nettoyage

In [ ]:
total_minutes = (time.time() - NOTEBOOK_START) / 60
print(f"Durée totale du notebook (GPU allumé) : {total_minutes:.0f} min")
for architecture, seconds in gpu_seconds.items():
    print(f"  entraînement {architecture} : {seconds / 60:.0f} min")
shutil.rmtree(DATASETS_DIR, ignore_errors=True)
shutil.rmtree(CLONE_DIR, ignore_errors=True)
print(sorted(path.name for path in OUTPUT_DIR.iterdir()))